In [6]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("ChurnPipeline")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())
print(spark.version)

4.2.0


In [7]:
b = spark.read.parquet("../bronze/customers")
bt = spark.read.parquet("../bronze/tickets")

print("Blank TotalCharges:", b.filter(F.trim("TotalCharges") == "").count())
print("Duplicate customerIDs:", b.count() - b.dropDuplicates(["customerID"]).count())
print("Null customerIDs:", b.filter(F.col("customerID").isNull()).count())
b.groupBy("MultipleLines").count().show()
b.groupBy("OnlineSecurity").count().show()

Blank TotalCharges: 11
Duplicate customerIDs: 0
Null customerIDs: 0
+----------------+-----+
|   MultipleLines|count|
+----------------+-----+
|              No| 3390|
|             Yes| 2971|
|No phone service|  682|
+----------------+-----+

+-------------------+-----+
|     OnlineSecurity|count|
+-------------------+-----+
|                 No| 3498|
|No internet service| 1526|
|                Yes| 2019|
+-------------------+-----+



In [8]:
log = []
def record(step, df):
    n = df.count()
    log.append((step, n))
    print(f"{step}: {n}")

record("bronze rows", b)

deduped = b.dropDuplicates(["customerID"])
record("after dedup", deduped)

cleaned = (deduped
  .withColumn("TotalCharges",   F.expr("try_cast(trim(TotalCharges) as double)"))
  .withColumn("MonthlyCharges", F.expr("try_cast(MonthlyCharges as double)"))
  .withColumn("tenure",         F.expr("try_cast(tenure as int)"))
  .withColumn("SeniorCitizen",  F.when(F.col("SeniorCitizen") == "1", "Yes").otherwise("No"))
  .replace(["No internet service", "No phone service"], "No")
  .withColumn("churn_flag",     F.when(F.col("Churn") == "Yes", 1).otherwise(0))
  .fillna({"TotalCharges": 0.0})      # tenure=0 customers have not been billed yet
)

is_valid = F.coalesce(
    F.col("customerID").isNotNull()
    & (F.col("MonthlyCharges") > 0)
    & (F.col("tenure") >= 0), F.lit(False))

valid   = cleaned.filter(is_valid)
invalid = cleaned.filter(~is_valid)
record("valid rows", valid)
record("rejected rows", invalid)

valid.write.mode("overwrite").parquet("../silver/customers")
invalid.write.mode("overwrite").parquet("../silver/customers_rejected")

bronze rows: 7043
after dedup: 7043
valid rows: 7043
rejected rows: 0


In [9]:
tix = (bt
  .dropDuplicates(["ticket_id"])
  .withColumn("ticket_id",      F.expr("try_cast(ticket_id as int)"))
  .withColumn("ticket_date",    F.expr("try_cast(ticket_date as date)"))
  .withColumn("resolution_hrs", F.expr("try_cast(resolution_hrs as int)"))
  .withColumn("category",       F.lower(F.trim("category")))
  .filter(F.col("customerID").isNotNull() & F.col("ticket_date").isNotNull()
          & (F.col("resolution_hrs") > 0))
)
record("tickets clean", tix)
tix.write.mode("overwrite").parquet("../silver/tickets")

tickets clean: 15000


In [10]:
spark.createDataFrame(log, ["step", "row_count"]).show(truncate=False)
spark.read.parquet("../silver/customers").printSchema()

+-------------+---------+
|step         |row_count|
+-------------+---------+
|bronze rows  |7043     |
|after dedup  |7043     |
|valid rows   |7043     |
|rejected rows|0        |
|tickets clean|15000    |
+-------------+---------+

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: string (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: 